Source:
* https://www.datacamp.com/tutorial/knowledge-graph-rag
* https://www.npmjs.com/package/download-git-repo
* https://github.com/tomasonjo/blogs/blob/master/llm/enhancing_rag_with_graph.ipynb?ref=blog.langchain.dev

## PROCESS

1. Load files from repo
2. Create function chunking from each file
    Metadata:
        filename: ?
        line_number_mapping: ?
        function:
        function_call_stack
    Content:
3. Chunking
    TextSplitter
    https://python.langchain.com/docs/how_to/code_splitter/

STEP 2: Initialize language model
1. instantiate language model (OpenAI)
2. llm.transformer.convert_to_graph_documents

STEP 3: Store to vector database
Embeddings

STEP 4: Retrieve knowledge for RAG

Step 5: Evaluate response (langchain)

In [3]:
! pip install --upgrade --quiet pip

In [36]:
! pip install "pinecone[grpc]"

  Using cached googleapis_common_protos-1.66.0-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached pinecone_plugin_inference-3.0.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached protoc_gen_openapiv2-0.0.1-py3-none-any.whl.metadata (1.5 kB)
Using cached googleapis_common_protos-1.66.0-py2.py3-none-any.whl (221 kB)
Using cached pinecone_plugin_inference-3.0.0-py3-none-any.whl (87 kB)
Using cached protoc_gen_openapiv2-0.0.1-py3-none-any.whl (7.9 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
  Attempting uninstall: pinecone-plugin-inference
    Found existing installation: pinecone-plugin-inference 1.1.0
    Uninstalling pinecone-plugin-inference-1.1.0:
      Successfully uninstalled pinecone-plugin-inference-1.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following 

In [47]:
pip install --upgrade pinecone pinecone-plugin-assistant


  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.32.3-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.75 requires requests_mock, which is not installed.
conda-repo-cli 1.0.75 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.75 requires requests==2.31.0, but you have requests 2.32.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [34]:
! pip install GitPython
! pip install pinecone-plugin-inference
! pip install langchain 
! pip install langchain-community
! pip install langchain_pinecone 
! pip install langchain-text-splitters
! pip install openai
! pip install pinecone
! pip install pinecone-client
! pip install pygithub 
! pip install python-dotenv
! pip install requests
! pip install streamlit
! pip install tiktoken
! pip install tree-sitter
! pip install tree-sitter-language-pack


  Using cached pinecone_plugin_inference-3.0.0-py3-none-any.whl.metadata (2.2 kB)
Using cached pinecone_plugin_inference-3.0.0-py3-none-any.whl (87 kB)
  Attempting uninstall: pinecone-plugin-inference
    Found existing installation: pinecone-plugin-inference 1.1.0
    Uninstalling pinecone-plugin-inference-1.1.0:
      Successfully uninstalled pinecone-plugin-inference-1.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pinecone-client 5.0.1 requires pinecone-plugin-inference<2.0.0,>=1.0.3, but you have pinecone-plugin-inference 3.0.0 which is incompatible.
  Using cached pinecone_plugin_inference-1.1.0-py3-none-any.whl.metadata (2.2 kB)
Using cached pinecone_plugin_inference-1.1.0-py3-none-any.whl (85 kB)
  Attempting uninstall: pinecone-plugin-inference
    Found existing installation: pinecone-plugin-inference 3.0.0
    Uninstalling pinecone-plugin-infer

# Constants

In [6]:
IGNORED_DIRS = {
    'node_modules', 
    'venv', 
    'env', 
    'dist', 
    'build', 
    'vendor',
    '__pycache__', 
    'pacakge.json',
    'tsconfig.json'
}

MAX_TOKENS_CHUNK_SIZE = 5000

In [7]:
class LanguageSupport:
    # Class-level dictionary to store file extensions and their languages
    lang_map = {
        ".cpp": "cpp",
        ".go": "go",
        ".java": "java",
        ".kt": "kotlin",
        ".js": "js",
        ".jsx": "js",
        ".ts": "ts",
        ".tsx": "ts",
        ".php": "php",
        ".proto": "proto",
        ".py": "python",
        ".rst": "rst",
        ".rb": "ruby",
        ".rs": "rust",
        ".scala": "scala",
        ".swift": "swift",
        ".md": "markdown",
        ".tex": "latex",
        ".html": "html",
        ".sol": "sol",
        ".cs": "csharp",
        ".cobol": "cobol",
        ".c": "c",
        ".lua": "lua",
        ".pl": "perl",
        ".hs": "haskell",
        ".ipynb": "ipynb"   # Not supported language for codeSplitter
    }

    @classmethod
    def is_supported_language(cls, extension):
        """Check if the given file extension is supported."""
        return extension in cls.lang_map

    @classmethod
    def get_language(cls, extension):
        """Get the language associated with a file extension."""
        return cls.lang_map.get(extension, "Unsupported language")

    @classmethod
    def add_language(cls, extension, language):
        """Add a new file extension and associated language."""
        cls.lang_map[extension] = language

    @classmethod
    def remove_language(cls, extension):
        """Remove a file extension from the mapping."""
        if extension in cls.lang_map:
            del cls.lang_map[extension]


# Setting API keys or secrets

In [9]:
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv()

pinecone_api_key = os.getenv('PINECONE_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
neo4j_uri = os.getenv('NEO4J_URI')
neo4j_username = os.getenv('NEO4J_USERNAME')
neo4j_password = os.getenv('NEO4J_PASSWORD')
repr(neo4j_uri)

"'neo4j+s://9f6c4287.databases.neo4j.io'"

# Clone a github repo locally

In [11]:
import os
import re
import json
import requests
from git import Repo, GitCommandError
from langchain.schema import Document


class GithubRepoData:
    def __init__(self, url: str):
        """
        Initializes the GithubRepoData object with repository details.

        Args:
            url (str): The GitHub repository URL.
        """
        # Match the URL to extract the owner and repo name
        pattern = r"(?:https?://|git@)github\.com[:/](.+?)/(.+?)(?:/|\.git|$)"
        match = re.match(pattern, url)
        if not match:
            raise ValueError(f"Invalid GitHub URL: {url}")
        owner, repo = match.groups()

        # Fetch repository details from GitHub API
        api_url = f'https://api.github.com/repos/{owner}/{repo}'
        response = requests.get(api_url)
        if response.status_code == 200:
            github = response.json()
            visibility = github.get('visibility', 'unknown')

            if visibility == 'public':
                self.id = github.get('id')
                self.owner = github.get('owner').get('login')
                self.fullname = github.get('full_name')
                self.repo_name = github.get('name')
                self.main_branch = github.get('default_branch')
                self.description = github.get('description')
                self.events_url = github.get('events_url')
                self.dir_path = os.path.join(self.repo_name)
            else:
                raise ValueError(f'Repository {self.owner}/{self.repo_name} is not public.')
        elif response.status_code == 404:
            raise ValueError(f"Repository {owner}/{repo} not found.")
        elif response.status_code == 403:
            raise ValueError("GitHub API rate limit exceeded. Please try again later.")
        else:
            raise ValueError(f"Failed to fetch repository details. HTTP Status: {response.status_code}")

    def to_dict(self):
        return {
            "id": self.id,
            "owner": self.owner,
            "fullname": self.fullname,
            "repo_name": self.repo_name,
            "description": self.description,
            "events_url": self.events_url,
            "dir_path": self.dir_path,
            "url": f"https://github.com/{self.owner}/{self.repo_name}"
        }

    def exists(self) -> bool:
        """Checks if the repository is already cloned."""
        return os.path.exists(self.dir_path)

    def clone(self) -> bool:
        """Clones the repository into the specified local directory."""
        if self.exists():
            print(f"Repository {self.repo_name} already cloned at {self.dir_path}.")
            return True

        try:
            os.makedirs(self.dir_path, exist_ok=True)
            clone_url = f"https://github.com/{self.fullname}.git"
            print(f"Cloning {self.repo_name} into {self.dir_path}")
            Repo.clone_from(clone_url, self.dir_path)
            print(f"Repository {self.repo_name} cloned successfully.")
            return True
        except GitCommandError as e:
            raise ValueError(f"Failed to clone {self.repo_name}: {e}")


class LocalDirectoryFiles:
    def __init__(self, dir_path: str, repo_fullname: str, main_branch: str):
        """
        Initializes the LocalDirectoryFiles object.

        Args:
            dir_path (str): Path to the local directory.
            repo_fullname (str): Full name of the repository (e.g., owner/repo).
            main_branch (str): Main branch of the repository.
        """
        self.dir_path = dir_path
        self.repo_fullname = repo_fullname
        self.main_branch = main_branch

    def read_file(self, filepath: str) -> str:
        """Reads the contents of a file."""
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                return f.read()
        except Exception as e:
            raise ValueError(f"Failed to read file {filepath}: {e}")

    def get_files(self) -> list[Document]:
        """
        Recursively retrieves all files in the directory and creates `Document` objects.

        Returns:
            list[Document]: List of Document objects.
        """
        documents = []
        for root, _, files in os.walk(self.dir_path):
            if any(ignored_dir in root for ignored_dir in IGNORED_DIRS):
                continue

            for file in files:
                file_path = os.path.join(root, file)
                extension = os.path.splitext(file_path)[1]
                print('extension: ', extension)
                if LanguageSupport.is_supported_language(extension) :
                    content = self.read_file(file_path)
                    relative_path = os.path.relpath(file_path, self.dir_path)
                    if content:
                        document = Document(
                            page_content=content,
                            metadata={
                                "filename": file,
                                "path": file_path,
                                "url": f"https://github.com/{self.repo_fullname}/blob/{self.main_branch}/{relative_path}"
                            }
                        )
                        documents.append(document)
        return documents

In [12]:
# TODO: Add persistance on already downloaded github repo
REPOS = {}
# TODO: Add persistence on already indexed github repo
REPO_LOCAL_DIRECTORIES = {}

In [13]:
REPOS

{}

In [14]:
repo1 = GithubRepoData("https://github.com/CoderAgent/SecureAgent")
if repo1.fullname not in REPOS:
    REPOS[repo1.fullname] = repo1
    if repo1.fullname not in REPO_LOCAL_DIRECTORIES:
        REPO_LOCAL_DIRECTORIES[repo1.fullname] = LocalDirectoryFiles(repo1.dir_path, repo1.fullname, repo1.main_branch)

repo2 = GithubRepoData('https://github.com/itancio/braintumor2')
if repo2.fullname not in REPOS:
    REPOS[repo2.fullname] = repo2
    if repo2.fullname not in REPO_LOCAL_DIRECTORIES:
        REPO_LOCAL_DIRECTORIES[repo2.fullname] = LocalDirectoryFiles(repo2.dir_path, repo2.fullname, repo2.main_branch)


localDirectoryFiles = REPO_LOCAL_DIRECTORIES['CoderAgent/SecureAgent']
documents = localDirectoryFiles.get_files()
for doc in documents:
    print(doc)
len(documents)


extension:  .ts
extension:  .ts
extension:  .ts
extension:  .ts
extension:  .ts
extension:  .ts
extension:  .ts
extension:  .ts
extension:  .ts
extension:  .ts
extension:  .ts
extension:  .ts
extension:  .ts
page_content='import { Octokit } from "@octokit/rest";
import { createNodeMiddleware } from "@octokit/webhooks";
import { WebhookEventMap } from "@octokit/webhooks-definitions/schema";
import * as http from "http";
import { App } from "octokit";
import { Review } from "./constants";
import { env } from "./env";
import { processPullRequest } from "./review-agent";
import { applyReview } from "./reviews";

// This creates a new instance of the Octokit App class.
const reviewApp = new App({
  appId: env.GITHUB_APP_ID,
  privateKey: env.GITHUB_PRIVATE_KEY,
  webhooks: {
    secret: env.GITHUB_WEBHOOK_SECRET,
  },
});

const getChangesPerFile = async (payload: WebhookEventMap["pull_request"]) => {
  try {
    const octokit = await reviewApp.getInstallationOctokit(
      payload.installa

13

# Parser/ Chunker

In [16]:

"""Chunker abstraction and implementations."""

import logging
import os
from abc import ABC, abstractmethod
from dataclasses import dataclass
from functools import cached_property
from typing import Any, Dict, List, Optional

import nbformat
import pygments
import tiktoken
from semchunk import chunk as chunk_via_semchunk
from tree_sitter import Node
from tree_sitter_language_pack import get_parser


tokenizer = tiktoken.get_encoding("cl100k_base")


class Chunk:
    @abstractmethod
    def content(self) -> str:
        """The content of the chunk to be indexed."""

    @abstractmethod
    def metadata(self) -> Dict:
        """Metadata for the chunk to be indexed."""


@dataclass
class FileChunk(Chunk):
    """A chunk of code or text extracted from a file in the repository."""

    file_content: str  # The content of the entire file, not just this chunk.
    file_metadata: Dict  # Metadata of the entire file, not just this chunk.
    start_byte: int
    end_byte: int

    @cached_property
    def filename(self):
        if not "file_path" in self.file_metadata:
            raise ValueError("file_metadata must contain a 'file_path' key.")
        return self.file_metadata["file_path"]

    @cached_property
    def content(self) -> Optional[str]:
        """The text content to be embedded. Might contain information beyond just the text snippet from the file."""
        return self.filename + "\n\n" + self.file_content[self.start_byte : self.end_byte]

    @cached_property
    def metadata(self):
        """Converts the chunk to a dictionary that can be passed to a vector store."""
        # Some vector stores require the IDs to be ASCII.
        filename_ascii = self.filename.encode("ascii", "ignore").decode("ascii")
        chunk_metadata = {
            # Some vector stores require the IDs to be ASCII.
            "id": f"{filename_ascii}_{self.start_byte}_{self.end_byte}",
            "start_byte": self.start_byte,
            "end_byte": self.end_byte,
            "length": self.end_byte - self.start_byte,
            # Note to developer: When choosing a large chunk size, you might exceed the vector store's metadata
            # size limit. In that case, you can simply store the start/end bytes above, and fetch the content
            # directly from the repository when needed.
            TEXT_FIELD: self.content,
        }
        chunk_metadata.update(self.file_metadata)
        return chunk_metadata

    @cached_property
    def num_tokens(self):
        """Number of tokens in this chunk."""
        return len(tokenizer.encode(self.content, disallowed_special=()))

    def __eq__(self, other):
        if isinstance(other, Chunk):
            return (
                self.filename == other.filename
                and self.start_byte == other.start_byte
                and self.end_byte == other.end_byte
            )
        return False

    def __hash__(self):
        return hash((self.filename, self.start_byte, self.end_byte))


class Chunker(ABC):
    """Abstract class for chunking a datum into smaller pieces."""

    @abstractmethod
    def chunk(self, content: Any, metadata: Dict) -> List[Chunk]:
        """Chunks a datum into smaller pieces."""


class CodeFileChunker(Chunker):
    """Splits a code file into chunks of at most `max_tokens` tokens each."""

    def __init__(self, max_tokens: int):
        self.max_tokens = max_tokens
        self.text_chunker = TextFileChunker(max_tokens)

    @staticmethod
    def _get_language_from_filename(filename: str):
        """Returns a canonical name for the language of the file, based on its extension.
        Returns None if the language is unknown to the pygments lexer.
        """
        # pygments doesn't recognize .tsx files and returns None. So we need to special-case them.
        extension = os.path.splitext(filename)[1]
        if extension == ".tsx":
            return "tsx"

        try:
            lexer = pygments.lexers.get_lexer_for_filename(filename)
            return lexer.name.lower()
        except pygments.util.ClassNotFound:
            return None

    def _chunk_node(self, node: Node, file_content: str, file_metadata: Dict) -> List[FileChunk]:
        """Splits a node in the parse tree into a flat list of chunks."""
        node_chunk = FileChunk(file_content, file_metadata, node.start_byte, node.end_byte)

        if node_chunk.num_tokens <= self.max_tokens:
            return [node_chunk]

        if not node.children:
            # This is a leaf node, but it's too long. We'll have to split it with a text tokenizer.
            return self.text_chunker.chunk(file_content[node.start_byte : node.end_byte], file_metadata)

        chunks = []
        for child in node.children:
            chunks.extend(self._chunk_node(child, file_content, file_metadata))

        for chunk in chunks:
            # This should always be true. Otherwise there must be a bug in the code.
            assert chunk.num_tokens <= self.max_tokens

        # Merge neighboring chunks if their combined size doesn't exceed max_tokens. The goal is to avoid pathologically
        # small chunks that end up being undeservedly preferred by the retriever.
        merged_chunks = []
        for chunk in chunks:
            if not merged_chunks:
                merged_chunks.append(chunk)
            elif merged_chunks[-1].num_tokens + chunk.num_tokens < self.max_tokens - 50:
                # There's a good chance that merging these two chunks will be under the token limit. We're not 100% sure
                # at this point, because tokenization is not necessarily additive.
                merged = FileChunk(
                    file_content,
                    file_metadata,
                    merged_chunks[-1].start_byte,
                    chunk.end_byte,
                )
                if merged.num_tokens <= self.max_tokens:
                    merged_chunks[-1] = merged
                else:
                    merged_chunks.append(chunk)
            else:
                merged_chunks.append(chunk)
        chunks = merged_chunks

        for chunk in merged_chunks:
            # This should always be true. Otherwise there's a bug worth investigating.
            assert chunk.num_tokens <= self.max_tokens

        return merged_chunks

    @staticmethod
    def is_code_file(filename: str) -> bool:
        """Checks whether pygment & tree_sitter can parse the file as code."""
        language = CodeFileChunker._get_language_from_filename(filename)
        return language and language not in ["text only", "None"]

    @staticmethod
    def parse_tree(filename: str, content: str) -> List[str]:
        """Parses the code in a file and returns the parse tree."""
        language = CodeFileChunker._get_language_from_filename(filename)

        if not language or language in ["text only", "None"]:
            logging.debug("%s doesn't seem to be a code file.", filename)
            return None

        try:
            parser = get_parser(language)
        except LookupError:
            logging.debug("%s doesn't seem to be a code file.", filename)
            return None
        # This should never happen unless there's a bug in the code, but we'd rather not crash.
        except Exception as e:
            logging.warn("Failed to get parser for %s: %s", filename, e)
            return None

        tree = parser.parse(bytes(content, "utf8"))

        if not tree.root_node.children or tree.root_node.children[0].type == "ERROR":
            logging.warning("Failed to parse code in %s.", filename)
            return None
        return tree

    def chunk(self, content: Any, metadata: Dict) -> List[Chunk]:
        """Chunks a code file into smaller pieces."""
        file_content = content
        file_metadata = metadata
        file_path = metadata["file_path"]

        if not file_content.strip():
            return []

        tree = self.parse_tree(file_path, file_content)
        if tree is None:
            return []

        file_chunks = self._chunk_node(tree.root_node, file_content, file_metadata)
        for chunk in file_chunks:
            # Make sure that the chunk has content and doesn't exceed the max_tokens limit. Otherwise there must be
            # a bug in the code.
            assert (
                chunk.num_tokens <= self.max_tokens
            ), f"Chunk size {chunk.num_tokens} exceeds max_tokens {self.max_tokens}."

        return file_chunks


class TextFileChunker(Chunker):
    """Wrapper around semchunk: https://github.com/umarbutler/semchunk."""

    def __init__(self, max_tokens: int):
        self.max_tokens = max_tokens
        self.count_tokens = lambda text: len(tokenizer.encode(text, disallowed_special=()))

    def chunk(self, content: Any, metadata: Dict) -> List[Chunk]:
        """Chunks a text file into smaller pieces."""
        file_content = content
        file_metadata = metadata
        file_path = file_metadata["file_path"]

        # We need to allocate some tokens for the filename, which is part of the chunk content.
        extra_tokens = self.count_tokens(file_path + "\n\n")
        text_chunks = chunk_via_semchunk(file_content, self.max_tokens - extra_tokens, self.count_tokens)

        file_chunks = []
        start = 0
        for text_chunk in text_chunks:
            # This assertion should always be true. Otherwise there's a bug worth finding.
            assert self.count_tokens(text_chunk) <= self.max_tokens - extra_tokens

            # Find the start/end positions of the chunks.
            start = file_content.index(text_chunk, start)
            if start == -1:
                logging.warning("Couldn't find semchunk in content: %s", text_chunk)
            else:
                end = start + len(text_chunk)
                file_chunks.append(FileChunk(file_content, file_metadata, start, end))

            start = end

        return file_chunks


class IpynbFileChunker(Chunker):
    """Extracts the python code from a Jupyter notebook, removing all the boilerplate.

    Based on https://github.com/GoogleCloudPlatform/generative-ai/blob/main/language/code/code_retrieval_augmented_generation.ipynb
    """

    def __init__(self, code_chunker: CodeFileChunker):
        self.code_chunker = code_chunker

    def chunk(self, content: Any, metadata: Dict) -> List[Chunk]:
        filename = metadata["file_path"]

        if not filename.lower().endswith(".ipynb"):
            logging.warn("IPYNBChunker is only for .ipynb files.")
            return []

        notebook = nbformat.reads(content, as_version=nbformat.NO_CONVERT)
        python_code = "\n".join([cell.source for cell in notebook.cells if cell.cell_type == "code"])

        tmp_metadata = {"file_path": filename.replace(".ipynb", ".py")}
        chunks = self.code_chunker.chunk(python_code, tmp_metadata)

        for chunk in chunks:
            # Update filenames back to .ipynb
            chunk.metadata["file_path"] = filename
        return chunks


ModuleNotFoundError: No module named 'semchunk'

In [17]:
class UniversalFileChunker(Chunker):
    """Chunks a file into smaller pieces, regardless of whether it's code or text."""

    def __init__(self, max_tokens: int):
        print('here')
        self.max_tokens = max_tokens
        self.code_chunker = CodeFileChunker(max_tokens)
        self.ipynb_chunker = IpynbFileChunker(self.code_chunker)
        self.text_chunker = TextFileChunker(max_tokens)

    def chunk(self, content: Any, metadata: Dict) -> List[Chunk]:
        if not "file_path" in metadata:
            raise ValueError("metadata must contain a 'file_path' key.")
        file_path = metadata["file_path"]
        
        # Figure out the appropriate chunker to use.
        if file_path.lower().endswith(".ipynb"):
            chunker = self.ipynb_chunker
        elif CodeFileChunker.is_code_file(file_path):
            chunker = self.code_chunker
        else:
            chunker = self.text_chunker
        chunk = chunker.chunk(content, metadata)
        
        return chunk
chunk = UniversalFileChunker(4000)

chunk.chunk("export const fles = () => {console.log('nothing')}", {'file_path': 'filename.ts'})

NameError: name 'Chunker' is not defined

In [19]:
! pip install --quiet langchain-text-splitters
! pip install --quiet langchain_experimental langchain_openai

In [20]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)
import json
from langchain.schema import Document


class BaseChunkingStrategy:
    """Base class with shared chunk size, overlap, and a default splitter."""
    chunk_size = 1024
    chunk_overlap = 200
    name = "BaseChunkingStrategy"

    @classmethod
    def splitter(cls):
        """Lazily initialize and return the default splitter."""
        return CharacterTextSplitter(
            chunk_size=cls.chunk_size,
            chunk_overlap=cls.chunk_overlap,
            add_start_index=True
        )

    @classmethod
    def create_documents(cls, doc):
        content = doc.page_content
        metadata = doc.metadata
        """Split text using the default splitter."""
        return cls.splitter().create_documents(
            [content], 
            [metadata]
        )


class SemanticChunkingStrategy(BaseChunkingStrategy):
    """Semantic chunking strategy with a shared splitter."""
    def __init__(self):
        self.name = "SemanticChunkingStrategy"
        self.splitter = SemanticChunker(
            OpenAIEmbeddings(),
            breakpoint_threshold_type="percentile",
            add_start_index=True
        )

    def create_documents(self, doc):
        content = doc.page_content
        metadata = doc.metadata
        try:
            return self.splitter.create_documents(
                [content], 
                [metadata]
            )
        except Exception:
            # Fallback to default splitter
            return BaseChunkingStrategy.create_documents(doc)


class CodeChunkingStrategy(BaseChunkingStrategy):
    """Code chunking strategy with instance-specific language."""
    def __init__(self, language=None):
        self.name = "CodeChunkingStrategy"
        self.language = language
        if language:
            self.splitter = RecursiveCharacterTextSplitter.from_language(
                language=self.language,
                chunk_size=BaseChunkingStrategy.chunk_size,
                chunk_overlap=BaseChunkingStrategy.chunk_overlap,
                add_start_index=True
            )
        else:
            self.splitter = BaseChunkingStrategy.splitter()

    def create_documents(self, doc):
        content = doc.page_content
        metadata = doc.metadata
        try:
            return self.splitter.create_documents(
                [content], 
                [metadata]
            )
        except Exception:
            # Fallback to default splitter
            return BaseChunkingStrategy.create_documents(doc)


class IpynbChunkingStrategy(BaseChunkingStrategy):
    """Chunking strategy for Python Notebook."""
    
    def __init__(self):
        self.name = "IpynbChunkingStrategy"
        self.code_splitter = CodeChunkingStrategy(language="python")

    def preprocess(self, doc):
        metadata = doc.metadata
        content = doc.page_content
        json_data = json.loads(content)
        cells = json_data.get('cells', [])

        # Separate code and markdown cells
        code_cells = ['\n'.join(cell['source']) for cell in cells if cell['cell_type'] == 'code']
        markdown_cells = ['\n'.join(cell['source']) for cell in cells if cell['cell_type'] in {'markdown', 'raw'}]

        return {
            "code": Document(page_content='\n'.join(code_cells), metadata=metadata),
            "markdown": Document(page_content='\n'.join(markdown_cells), metadata=metadata)
        }

    def create_documents(self, doc):
        processed_docs = self.preprocess(doc)
        code_doc = processed_docs['code']
        markdown_doc = processed_docs['markdown']

        try:
            # Split code using CodeChunkingStrategy
            code_chunks = self.code_splitter.create_documents(code_doc)
            # Use BaseChunkingStrategy for markdown
            markdown_chunks = BaseChunkingStrategy.create_documents(markdown_doc)
            return code_chunks + markdown_chunks
        except Exception:
            # Fallback to default splitter
            return BaseChunkingStrategy.create_documents(doc)


class ChunkingManager:
    """Manager to select appropriate chunking strategy."""
    def __init__(self):
        self.code_splitter = lambda lang: CodeChunkingStrategy(lang)
        self.ipynb_splitter = IpynbChunkingStrategy()
        self.semantic_splitter = SemanticChunkingStrategy()

    def get_splitter(self, language):
        """Return the appropriate splitter based on file type."""
        if language == "ipynb":
            splitter = self.ipynb_splitter
            print(f"Splitter adopted: {splitter.name}")
            return splitter
        elif language:
            splitter = self.code_splitter(language)
            print(f"Splitter adopted: {splitter.name} for language {language}")
            return splitter
        else:
            splitter = self.semantic_splitter
            print(f"Splitter adopted: {splitter.name} (default for unsupported file type)")
            return splitter

    def create_documents(self, doc):
        metadata = doc.metadata
        filename = metadata.get('filename')
        extension = os.path.splitext(filename)[1]
        if LanguageSupport.is_supported_language(extension):
            lang = LanguageSupport.get_language(extension)
        else:
            lang = None
        print('language: ', lang)
        """Create documents using the appropriate splitter."""
        splitter = self.get_splitter(lang)
        return splitter.create_documents(doc)

ImportError: cannot import name 'version_short' from 'pydantic.version' (/opt/anaconda3/lib/python3.11/site-packages/pydantic/version.cpython-311-darwin.so)

In [ ]:
chunker = ChunkingManager()
chunks = chunker.create_documents(documents[1])
chunks

# Embedding


In [ ]:
%pip install --upgrade --quiet  langchain langchain-community langchain-openai langchain-experimental neo4j wikipedia tiktoken yfiles_jupyter_graphs

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings()


def embed_content(content):
    """Embeds documents using the OpenAI embeddings model."""
    
    embeddings = embeddings_model.embed_documents(content)
    return embeddings

In [ ]:
chunker = ChunkingManager()
embeddings = []
chunks = []
for document in documents:
    chunked_docs = chunker.create_documents(document)
    chunks.extend(chunked_docs)
    for chunk in chunked_docs:
        content = chunk.page_content
        
        embedded_content = embed_content(content)
        embeddings.append(embedded_content)
        

In [ ]:
# Temporarily write embeddings and chunks in a file

with open('embeddings.txt', 'w') as f:
    f.write(f'{embeddings}')
with open('chunks.txt', 'w') as f:
    f.write(f'{chunks}')

# Temporarily read embeddings and chunks in a file
with open('embeddings.txt', 'r') as f:
    embeddings_f = f.read().strip()
with open('chunks.txt', 'r') as f:
    chunks_f = f.read().strip()

In [ ]:
print('embeddings: ',len(embeddings), len(embeddings[0]))
print('chunks: ',len(chunks), len(chunks[0]))

print('embeddings: ',len(embeddings_f), len(embeddings_f[0]))
print('chunks: ',len(chunks_f), len(chunks_f[0]))

# Storing in the VectorStore

In [ ]:
! pip install pinecone-client langchain_pinecone

In [ ]:
from langchain_pinecone import PineconeVectorStore


pinecone = Pinecone(api_key=pinecone_api_key)

pinecone_store = PineconeVectorStore.from_documents(docs, embeddings_model, index_name='index')


# GRAPH RAG

In [ ]:
%pip install neo4j yfiles_jupyter_graphs

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI

graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password)
llm = ChatOpenAI(
    temperature=0, 
    model_name="gpt-3.5-turbo-0125") # gpt-4-0125-preview occasionally has issues
llm_transformer = LLMGraphTransformer(llm=llm)

graph_documents = llm_transformer.convert_to_graph_documents(chunks)
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

In [ ]:
# directly show the graph resulting from the given Cypher query
default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"

def showGraph(cypher: str = default_cypher):
    # create a neo4j session to run queries
    driver = GraphDatabase.driver(
        uri = neo4j_uri,
        auth = (neo4j_username,
                neo4j_password))
    session = driver.session()
    widget = GraphWidget(graph = session.run(cypher).graph())
    widget.node_label_mapping = 'id'
    #display(widget)
    return widget

showGraph()

# Storing in the the vectorstore

# Retrieval and reranking using nvidia

In [25]:
from langchain_community.vectorstores import Neo4jVector

# Unstructured data retriever
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(),
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)


In [26]:
# Retriever
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from typing import Tuple, List, Optional

graph.query(
    "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

entity_chain = prompt | llm.with_structured_output(Entities)

/Users/it/Desktop/Proj_CodebaseRag2/venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3577: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


In [27]:
# Test
entity_chain.invoke({"question": "What methods do you have"}).names

['methods']

In [28]:
def generate_full_text_query(input: str) -> str:
    """
    Generate a full-text search query for a given input string.

    This function constructs a query string suitable for a full-text search.
    It processes the input string by splitting it into words and appending a
    similarity threshold (~2 changed characters) to each word, then combines
    them using the AND operator. Useful for mapping entities from user questions
    to database values, and allows for some misspelings.
    """
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

# Fulltext index query
def structured_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned
    in the question
    """
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [29]:
print(structured_retriever("what is Reviewchanges "))

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (node, node) { ... }} {position: line: 3, column: 13, offset: 104} for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node,score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n              MATCH (node)<-[r:!MENTIONS]-(neighbor)\n              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output\n            }\n            RETURN output LIMIT 50\n            "


In [30]:
# Final retriever
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    return final_data

# Defining the RAG chain

In [38]:
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.output_parsers import StrOutputParser

# Condense a chat history and follow-up question into a standalone question
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""  # noqa: E501
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | ChatOpenAI(temperature=0)
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x : x["question"]),
)

In [39]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [43]:
chain.invoke({"question": "Show the reviewChanges"})

Search query: Show the reviewChanges


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (node, node) { ... }} {position: line: 3, column: 13, offset: 104} for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node,score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n              MATCH (node)<-[r:!MENTIONS]-(neighbor)\n              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output\n            }\n            RETURN output LIMIT 50\n            "
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNoti

'The reviewChanges function filters files, calculates token length, and divides patches based on size limits. It also calls other functions like buildPatchPrompt.'

In [34]:
chain.invoke({"Show the reviewChanges code"})

AttributeError: 'set' object has no attribute 'get'